# 🎛️ ComfyCluster: Worker

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlamandaCathartica/ComfyCluster/blob/main/Worker.ipynb)

Run this on each worker account (give each a different `WORKER_NAME`). It starts a headless ComfyUI behind a free Cloudflare quick tunnel and prints a **pairing code**. Paste the code into `Controller.ipynb`. You never open this tab's ComfyUI yourself.

Pick a **GPU runtime**, run the cell and leave it running. Models come from `models.json` in the repository. For gated models add a Colab secret named `HF_TOKEN`.

In [ ]:
#@title ▶️ Start ComfyCluster worker
import json, os, subprocess, sys, time, urllib.request

REPO_URL = 'https://github.com/AlamandaCathartica/ComfyCluster' #@param {type:"string"}
BRANCH = 'main' #@param {type:"string"}
WORKER_NAME = 'worker1' #@param {type:"string"}
USE_DRIVE = False #@param {type:"boolean"}

ROOT = '/content'
REPO_DIR = f'{ROOT}/ComfyCluster'
ENGINE = f'{REPO_DIR}/engine/worker.py'
CODE_FILE = f'{ROOT}/comfycluster_code.txt'
LOG_FILE = f'{ROOT}/comfycluster_worker.log'
CACHE_DIR = f'{ROOT}/_comfycluster_cache'

if USE_DRIVE:
  from google.colab import drive
  if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
  CACHE_DIR = '/content/drive/MyDrive/ComfyCluster/_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# ---- download / update the code from GitHub ----
subprocess.run(['pkill', '-f', 'engine/worker.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
  r = subprocess.run(f'git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD',
                     shell=True, capture_output=True, text=True)
else:
  subprocess.run(['rm', '-rf', REPO_DIR], check=False)
  r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO_DIR], capture_output=True, text=True)
if r.returncode != 0:
  print(r.stdout, r.stderr)
  if not os.path.exists(ENGINE):
    raise SystemExit(f'Could not download {REPO_URL} (branch {BRANCH}). Check the URL/branch and that the repo is public.')
  print('Update failed; using the copy already in this session.')
print(f"ComfyCluster v{open(f'{REPO_DIR}/VERSION').read().strip()} ready")

# ---- start the worker (installs ComfyUI, downloads models, opens a tunnel, makes the pairing code) ----
env = dict(os.environ)
try:
  from google.colab import userdata
  env['HF_TOKEN'] = userdata.get('HF_TOKEN')    # optional secret, only needed for gated models
except Exception:
  pass
if os.path.exists(CODE_FILE): os.remove(CODE_FILE)
cmd = [sys.executable, ENGINE, '--name', WORKER_NAME, '--root', ROOT, '--cache-dir', CACHE_DIR,
       '--models', f'{REPO_DIR}/models.json', '--code-file', CODE_FILE]
proc = subprocess.Popen(cmd, stdout=open(LOG_FILE, 'w'), stderr=subprocess.STDOUT, env=env, start_new_session=True)

def tail(n=1):
  try: return open(LOG_FILE).read().strip().splitlines()[-n:]
  except OSError: return []

from IPython.display import clear_output
shown = ''
print('Installing ComfyUI and starting the worker (the first run takes several minutes)...')
while not os.path.exists(CODE_FILE):
  if proc.poll() is not None:
    print(open(LOG_FILE).read()[-3000:]); raise SystemExit('The worker stopped. See the log above.')
  last = (tail() or [''])[0]
  if last != shown: shown = last; print(last)
  time.sleep(3)
CODE = open(CODE_FILE).read()

# ---- keep this cell running ----
def _status():
  try:
    q = json.load(urllib.request.urlopen('http://127.0.0.1:8188/queue', timeout=5))
    return f"running {len(q['queue_running'])} | pending {len(q['queue_pending'])}"
  except Exception as e:
    return f'ComfyUI not responding ({e})'
try:
  while proc.poll() is None:
    clear_output(wait=True)
    print(f'✅ Worker "{WORKER_NAME}" is ready. Paste this pairing code into the controller notebook:\n')
    print(CODE)
    print('\nKeep it private: anyone with the code can use this worker.')
    print('\n' + time.strftime('%H:%M:%S'), '-', _status())
    print('\nLeave this cell running. There is nothing else to do on this tab.')
    time.sleep(20)
  print(f'The worker exited. See {LOG_FILE} and {ROOT}/comfyui.log')
except KeyboardInterrupt:
  print('Cell stopped. The worker keeps running in the background.')

If the worker restarts it gets a new address: run it again, paste the new code into the controller and restart the controller cell.